# Sudoku Solver Comparison: Deterministic Backtracking vs. RL (DQN)

This notebook compares two approaches to solving Sudoku puzzles:

| Approach | Module | Key Idea |
|----------|--------|----------|
| **Backtracking** | `backtracking_solver.py` | Constraint propagation + recursive search with MRV heuristic |
| **Reinforcement Learning** | `rl_agent.py` | Deep Q-Network (DQN) trained to select cell/digit actions |

We evaluate both solvers on **easy**, **medium**, and **hard** puzzles across three axes:
1. **Correctness** – does the solver produce a valid, complete board?
2. **Speed** – wall-clock time per puzzle.
3. **Reliability** – success rate over multiple random puzzles.

In [ ]:
import sys, time, numpy as np
sys.path.insert(0, '..')  # allow imports from repo root

from sudoku_game import SudokuGame, SudokuValidator
from backtracking_solver import BacktrackingSolver
from rl_agent import SudokuRLAgent
from config import rl_config, DIFFICULTY_LEVELS

## 1. Helper Functions

We define a small benchmark harness that runs each solver on *N* random
puzzles of a given difficulty and records solve time, success, and steps.

In [ ]:
def benchmark_backtracking(difficulty: str, n_puzzles: int = 20):
    """Benchmark the deterministic backtracking solver."""
    solver = BacktrackingSolver()
    results = []
    for _ in range(n_puzzles):
        game = SudokuGame(difficulty)
        solved = solver.solve(game)
        valid = SudokuValidator.is_valid_board(game.board) if solved else False
        m = solver.get_metrics()
        results.append({
            'solved': solved and valid,
            'time': m['solve_time'],
            'steps': m['steps'],
            'backtracks': m['backtracks'],
        })
    return results


def benchmark_rl(difficulty: str, n_puzzles: int = 20,
                 max_steps: int = 200):
    """Benchmark the RL (DQN) solver (untrained weights)."""
    agent = SudokuRLAgent(device='cpu')
    results = []
    for _ in range(n_puzzles):
        game = SudokuGame(difficulty)
        start = time.perf_counter()
        steps = 0
        for step in range(max_steps):
            if game.is_complete():
                break
            valid_actions = agent.get_valid_actions(game)
            if not valid_actions:
                break
            state = game.get_encoded_state()
            row, col, digit = agent.select_action(
                state, valid_actions, training=False
            )
            game.place_digit(row, col, digit)
            steps += 1
        elapsed = time.perf_counter() - start
        solved = game.is_complete() and SudokuValidator.is_valid_board(game.board)
        results.append({
            'solved': solved,
            'time': elapsed,
            'steps': steps,
        })
    return results


def summarise(results, label):
    """Print a summary table row."""
    n = len(results)
    success = sum(1 for r in results if r['solved'])
    times = [r['time'] for r in results]
    steps = [r['steps'] for r in results]
    print(f"{label:30s}  success={success}/{n}  "
          f"avg_time={np.mean(times)*1000:8.2f} ms  "
          f"avg_steps={np.mean(steps):6.1f}")

## 2. Run Benchmarks

We solve 20 random puzzles per difficulty with each solver.

> **Note:** The RL agent here uses *untrained* (random) weights. With a
> trained model (`python train.py`), RL success rates improve substantially
> (see README for expected numbers). The backtracking solver always achieves
> 100 % because it is a complete search algorithm.

In [ ]:
N_PUZZLES = 20

for diff in ['easy', 'medium', 'hard']:
    print(f'\n--- {diff.upper()} (givens={DIFFICULTY_LEVELS[diff]}) ---')
    bt = benchmark_backtracking(diff, N_PUZZLES)
    rl = benchmark_rl(diff, N_PUZZLES)
    summarise(bt, f'Backtracking ({diff})')
    summarise(rl, f'RL / DQN     ({diff})')

## 3. Analysis

### Correctness

| Solver | Guarantee |
|--------|-----------|
| Backtracking | **Always correct** – explores the full search tree and returns a valid solution whenever one exists. |
| RL (DQN) | **No guarantee** – the agent greedily picks the highest-Q action; if the Q-network has not been well-trained the agent may place conflicting digits or get stuck. |

### Speed

Backtracking with constraint propagation solves most 9×9 puzzles in **< 5 ms**.
The RL agent must run a CNN forward pass for every single action, making it
orders of magnitude slower per puzzle.

### Reliability

Backtracking achieves a **100 %** success rate on any valid puzzle.
An untrained RL agent has a near-**0 %** success rate; even a well-trained
agent on easy puzzles plateaus around **~95 %** (see README).

### When is RL useful?

* **Learning & research** – understanding how neural networks can learn
  combinatorial reasoning.
* **Heuristic guidance** – a trained RL agent can rank candidate moves to
  accelerate a hybrid solver.
* **Generalisation** – RL may transfer learned patterns to variant puzzles
  (e.g., 16×16 Sudoku) without re-engineering the algorithm.

## 4. Conclusion

| Criterion | Backtracking | RL (DQN) |
|-----------|-------------|----------|
| Correctness | ✅ Guaranteed | ❌ Approximate |
| Speed | ✅ < 5 ms typical | ❌ ~seconds (CPU inference) |
| Success rate | ✅ 100 % | ⚠️ ~0–95 % depending on training |
| Adaptability | ❌ Algorithm-specific | ✅ Learns from data |
| Research value | Low | High |

For **production use**, the backtracking solver is the clear winner.
The RL approach is valuable for **research and education** in applying
deep reinforcement learning to combinatorial constraint problems.